In [24]:
from dataclasses import dataclass
from typing import Optional,Tuple,List
from collections import namedtuple

import math
import time
import json
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from transformers.modeling_utils import PreTrainedModel



def log(t, eps = 1e-20):
    return torch.log(t.clamp(min = eps))


def gumbel_noise(t):
    noise = torch.zeros_like(t).uniform_(0, 1)
    return -log(-log(noise))


def _topk(array, k:int):
    top_k_indices = torch.topk(array, k)[-1]
    one_hot_length = array.shape[-1]
    one_hot_indices = torch.nn.functional.one_hot(top_k_indices, one_hot_length).to(array.dtype)
    top_k_values = torch.einsum('...s,...is->...i', array, one_hot_indices)
    return top_k_values, top_k_indices, one_hot_indices.max(2)[0]


def one_hot_with_ignore(indices, num_classes, dtype=torch.int32):
    # 创建基础张量
    result = torch.zeros(indices.shape + (num_classes,), dtype=dtype, device=indices.device)
    # 将indices展平并转换为int64
    flat_indices = indices.reshape(-1).to(torch.int64)
    # 创建mask
    mask = (flat_indices >= 0) & (flat_indices < num_classes)
    # 使用where来替代布尔索引
    safe_indices = torch.where(mask, flat_indices, torch.zeros_like(flat_indices))
    # 重塑result为2D
    flat_result = result.reshape(-1, num_classes)
    # 使用scatter填充1
    flat_result.scatter_(1, safe_indices.unsqueeze(1), mask.unsqueeze(1).to(dtype))
    return result
    
def _load_balancing_loss(router_probs, expert_indices, mask=None) -> float:
  num_experts = router_probs.shape[-1]
  # Shape: [num_groups, tokens_per_group, num_selected_experts, num_experts].
  expert_mask = torch.nn.functional.one_hot(expert_indices, num_experts)
  # Shape: [num_groups, tokens_per_group, num_experts]
  expert_mask = expert_mask.max(2)[0].to(torch.float32)
  if mask is None:
      tokens_per_group_and_expert = expert_mask.mean(1)
      router_prob_per_group_and_expert = router_probs.mean(1)
  else:
      tokens_per_group_and_expert = expert_mask.sum(1) / mask.sum(1)
      router_prob_per_group_and_expert = router_probs.sum(1) / mask.sum(1)
  p = tokens_per_group_and_expert * router_prob_per_group_and_expert
  return p.mean() * num_experts**2

class MoeFeedForward(nn.Module):
    def __init__(self, config) -> None:
        super(MoeFeedForward, self).__init__()
        self.config = config
        self.dim = config.dim
        self.intermediate_size = config.intermediate_size
        self.mgate = config.mgate
        self.mgate_dim = config.mgate_dim
        self.num_experts = getattr(config, 'num_experts', 8)
        self.dtype = getattr(config, 'torch_dtype', torch.float16)
        self.int8 = getattr(config, 'int8', True)
        self.wi_gate_0 = nn.Parameter(torch.empty(self.num_experts, self.dim, self.intermediate_size))
        self.wi_0 = nn.Parameter(torch.empty(self.num_experts, self.dim, self.intermediate_size))
        self.wo_0 = nn.Parameter(torch.empty(self.num_experts, self.intermediate_size, self.dim))

        nn.init.constant_(self.wi_gate_0, 1.0)
        nn.init.constant_(self.wi_0, 1.0)
        nn.init.constant_(self.wo_0, 1.0)
        
        if self.mgate:
            self.mg = nn.Parameter(torch.empty(self.num_experts, self.dim, self.mgate_dim))
            nn.init.constant_(self.mg, 1.0)

        if self.int8:
            self.scales1 = nn.Parameter(torch.empty(self.num_experts, self.intermediate_size, dtype=torch.float16))
            self.scales3 = nn.Parameter(torch.empty(self.num_experts, self.intermediate_size, dtype=torch.float16))
            self.scales2 = nn.Parameter(torch.empty(self.num_experts, self.dim, dtype=torch.float16))
        
    def forward(self, expert_inputs, expert_index, compute_n_expert=0, training=False):
        if expert_inputs.shape[-2] == 1 and expert_inputs.shape[0] == 1:
            theta_wi = self.wi_0[expert_index]
            theta_wo = self.wo_0[expert_index]
            theta_wi_gated = self.wi_gate_0[expert_index]
            hidden0 = torch.einsum("gch,gehm->gem", expert_inputs, theta_wi)
            hidden1 = torch.einsum("gch,gehm->gem", expert_inputs, theta_wi_gated)
            hidden1 = torch.nn.functional.silu(hidden1)
            hidden = hidden1 * hidden0

            if self.mgate:
                assert isinstance(self.mgate_dim, int)
                inner_gate = self.mg[expert_index]
                mgate_scores = torch.einsum('gch,gehm->gem', expert_inputs, inner_gate)
                mgate_scores = torch.nn.functional.softmax(mgate_scores.to(torch.float32), dim=-1)
                mgate_scores = mgate_scores.to(self.dtype)
                G, E, H = hidden.shape
                hidden = hidden.reshape(G, E, self.mgate_dim, H // self.mgate_dim)
                hidden = torch.einsum('gei,geif->geif', mgate_scores, hidden)
                hidden = hidden.reshape(G, E, H)

            hidden = torch.einsum("gem,gemh->geh", hidden, theta_wo)
        else:
            theta_wi = self.wi_0[expert_index: expert_index + compute_n_expert]
            theta_wo = self.wo_0[expert_index: expert_index + compute_n_expert]
            theta_wi_gated = self.wi_gate_0[expert_index: expert_index + compute_n_expert]
            hidden0 = torch.einsum("gecm,emh->gech", expert_inputs, theta_wi)
            hidden1 = torch.einsum("gecm,emh->gech", expert_inputs, theta_wi_gated)
            hidden1 = torch.nn.functional.silu(hidden1)
            hidden = hidden1 * hidden0
            # expert_inputs: gecm,  mgatew: meh  -> 
            if self.mgate:
                assert isinstance(self.mgate_dim, int)
                inner_gate = self.mg[expert_index: expert_index + compute_n_expert]
                mgate_scores = torch.einsum('gecm,emi->geci', expert_inputs, inner_gate)
                mgate_scores = torch.nn.functional.softmax(mgate_scores.to(torch.float32), dim=-1)
                mgate_scores = mgate_scores.to(self.dtype)
                G, E, C, H = hidden.shape
                hidden = hidden.reshape(G, E, C, self.mgate_dim, H // self.mgate_dim)
                hidden = torch.einsum('geci,gecif->gecif', mgate_scores, hidden)
                hidden = hidden.reshape(G, E, C, H)
            hidden = torch.einsum("gech,ehm->gecm", hidden, theta_wo)
        return hidden

class MoeBlock(nn.Module):
    def __init__(self, config) -> None:
        super(MoeBlock, self).__init__()
        self.config = config
        self.MoeFeedForward = MoeFeedForward(config)
        self.min_group_size = 1
        self.num_experts = getattr(config, 'num_experts', 8)
        self.dim = getattr(config, 'dim', 4096)
        self.intermediate_size = getattr(config, 'intermediate_size', 5632)
        self.mgate_dim = getattr(config, 'mgate_dim', 44)
        self.topn = getattr(config, 'num_experts_per_tok', 2)
        self.expert_capacity_factor = getattr(config, 'expert_capacity_factor', 1.5)
        self.gate_noise_coef = getattr(config, 'gate_noise_coef', 0)
        self.sfm_after_topn = getattr(config, 'sfm_after_topn', True)
        self.dtype = getattr(config, 'torch_dtype', torch.float16)
        self.aux_loss_coef = getattr(config, 'aux_loss_coef', 0)
        self.router_z_loss_coef = getattr(config, 'router_z_loss_coef', 0)
        self.expert_chunk_size = getattr(config, 'expert_chunk_size', 1)


        self.router_gate = nn.Parameter(torch.empty(self.dim, self.num_experts))

    def forward(self, inputs, paddings=None):
        if inputs.shape[1] == 1:
            num_groups = inputs.shape[0]
            num_tokens = inputs.shape[0]*inputs.shape[1]
            # num_tokens = np.prod(inputs.shape[:-1]).to(inputs.device)
            tokens_per_group = num_tokens // num_groups
            assert num_tokens % num_groups == 0, print(f'‘num_tokens % num_groups -> {num_tokens} % {num_groups} != 0’')

            grouped_inputs = torch.reshape(inputs, (num_groups, tokens_per_group, self.dim))
            router_logits = torch.einsum('gsm,me->gse', grouped_inputs, self.router_gate)

            if self.gate_noise_coef > 0.0:
            #   print(f'gate_noise_coef: {self.gate_noise_coef}')
                noise = gumbel_noise(router_logits)
                router_logits += noise * self.gate_noise_coef
            # one_hot_indices: b l e  expert_index: b l topn
            _, expert_index, one_hot_indices = _topk(router_logits, k=self.topn)
        
            if self.sfm_after_topn:
                assert one_hot_indices is not None
                router_mask = (1 - one_hot_indices) * torch.finfo(self.dtype).min
                _router_logits = router_logits + router_mask
                router_probs = torch.nn.functional.softmax(_router_logits.to(torch.float32), dim=-1)
            else:
                # gse
                router_probs = torch.nn.functional.softmax(router_logits.to(torch.float32), dim=-1)
            
            
            router_probs = router_probs.to(self.dtype) # ble
            if paddings is not None:
                # the one means reserved in paddings
                gate_mask = torch.reshape(paddings, grouped_inputs.shape[:2])
                gate_mask = gate_mask.unsqueeze(-1) # bl1
                router_probs *= gate_mask # ble
            else:
                gate_mask = None

            if paddings is not None:
                expert_index *= (2 * gate_mask - 1) # lsp:masked expert set to negative, it would not bd considered when use function `one_hot_with_ignore`
                no_gate_mask = gate_mask - 1
                expert_index += no_gate_mask.repeat(1, 1, expert_index.shape[-1])
            
            # g * 2 * s
            expert_index = expert_index.permute(0, 2, 1)
            expert_index = expert_index.reshape(num_groups, -1)

            router_probs = router_probs.squeeze(1)
            _expert_outputs = self.MoeFeedForward(grouped_inputs, expert_index, router_probs)
            router_probs = torch.gather(router_probs, dim=1, index=expert_index)
            combined_outputs = torch.einsum('geh,ge->gh', _expert_outputs, router_probs)
            combined_outputs = combined_outputs.reshape(*inputs.shape)
            return combined_outputs
        else:
            num_groups = inputs.shape[0]
            num_tokens = inputs.shape[0]*inputs.shape[1]
            # num_tokens = np.prod(inputs.shape[:-1]).to(inputs.device)
            tokens_per_group = num_tokens // num_groups
            assert num_tokens % num_groups == 0, print(f'‘num_tokens % num_groups -> {num_tokens} % {num_groups} != 0’')
            # print(f'expert_capacity_factor: {self.expert_capacity_factor}')
            
            expert_capacity = math.ceil(self.expert_capacity_factor * tokens_per_group / self.num_experts)
            max_group_size = int(inputs.shape[1])
            expert_capacity = min(expert_capacity, max_group_size)
            expert_capacity = max(expert_capacity, self.min_group_size)
            # print(f'expert_capacity: {expert_capacity}')

            grouped_inputs = torch.reshape(inputs, (num_groups, tokens_per_group, self.dim))
            router_logits = torch.einsum('gsm,me->gse', grouped_inputs, self.router_gate)
            # __import__('ipdb').set_trace()

            if self.gate_noise_coef > 0.0:
            #   print(f'gate_noise_coef: {self.gate_noise_coef}')
                noise = gumbel_noise(router_logits)
                router_logits += noise * self.gate_noise_coef
            # one_hot_indices: b l e  expert_index: b l topn
            _, expert_index, one_hot_indices = _topk(router_logits, k=self.topn)
        
            if self.sfm_after_topn:
                assert one_hot_indices is not None
                router_mask = (1 - one_hot_indices) * torch.finfo(self.dtype).min
                _router_logits = router_logits + router_mask
                router_probs = torch.nn.functional.softmax(_router_logits.to(torch.float32), dim=-1)
            else:
                # gse
                router_probs = torch.nn.functional.softmax(router_logits.to(torch.float32), dim=-1)
            
            router_probs = router_probs.to(self.dtype) # ble
            if paddings is not None:
                # the one means reserved in paddings
                gate_mask = torch.reshape(paddings, grouped_inputs.shape[:2])
                gate_mask = gate_mask.unsqueeze(-1) # bl1
                router_probs *= gate_mask # ble
            else:
                gate_mask = None
        
            aux_loss, router_z_loss = 0.0, 0.0
            if self.aux_loss_coef is not None:
                aux_loss = _load_balancing_loss(router_probs, expert_index, gate_mask)
                aux_loss *= self.aux_loss_coef
                # print(f'aux_loss: {aux_loss}')

            if self.router_z_loss_coef is not None: 
                # The purpose is to prevent the output of the router from becoming too extreme or unstable, to ensure that 
                # the probability distribution is not concentrated on a very small number of experts, and to prevent excessively large logits.
                # <=> torch.logsumexp(logits, dim = -1)
                router_z_loss = torch.logsumexp(router_logits, dim = -1)
                router_z_loss = router_z_loss.square()            
                router_z_loss = self.router_z_loss_coef * router_z_loss.mean()
                # print(f'router_z_loss: {router_z_loss}')

            if paddings is not None:
                expert_index *= (2 * gate_mask - 1) # lsp:masked expert set to negative, it would not bd considered when use function `one_hot_with_ignore`
                no_gate_mask = gate_mask - 1
                expert_index += no_gate_mask.repeat(1, 1, expert_index.shape[-1])
            
            aux_loss = aux_loss + router_z_loss
            # g * 2 * s
            expert_index = expert_index.permute(0, 2, 1)
            # g * 2s
            expert_index = expert_index.reshape(num_groups, -1)
            # g * 2s * e, expert_index , this function can ignore negative
            expert_mask = one_hot_with_ignore(expert_index, self.num_experts, dtype=torch.int32)
            # # g * 2s * e 
            token_priority = torch.cumsum(expert_mask, dim=1) * expert_mask - 1.0
            # # g * 2 * s * e
            token_priority = token_priority.reshape(num_groups, self.topn, -1, self.num_experts)
            # # g * s * 2 * e  lsp: per token select 2 expert，expert corresponss to position value mean current rank expert selected token numbers
            token_priority = token_priority.permute(0, 2, 1, 3)
            token_priority = token_priority.max(2)[0].to(torch.int32) # (b*l) * e
            if self.expert_chunk_size is None:
                compute_n_expert = self.num_experts
            else:
                compute_n_expert = self.num_experts // self.expert_chunk_size
                assert self.num_experts % self.expert_chunk_size == 0, print(self.num_experts, self.expert_chunk_size)
            combined_outputs = None
            # print(f'compute_n_expert: {compute_n_expert}')

            for expert_index in range(0, token_priority.shape[-1], compute_n_expert):
                _token_priority = token_priority[..., expert_index: expert_index+compute_n_expert]
                _router_probs = router_probs[..., expert_index: expert_index+compute_n_expert].to(self.dtype)
                # lsp： _dispatch_mask: (g*s)ec
                _dispatch_mask = one_hot_with_ignore(_token_priority.reshape(-1, _token_priority.shape[-1]), expert_capacity, dtype=torch.int32)
                _dispatch_mask = _dispatch_mask.reshape(num_groups, tokens_per_group, compute_n_expert, -1).to(self.dtype).to(_router_probs.device)
                _combine_array = torch.einsum('gse,gsec->gsec', _router_probs, _dispatch_mask)
                _combine_array = _combine_array.to(self.dtype)
                # expert inputs mask：gsm x gsec -> gecm，  _dispatch_mask can drop unused token
                _expert_inputs = torch.einsum('gsd,gsec->gecd', grouped_inputs, _dispatch_mask)
                # g * e * c * m
                # _expert_outputs = self._call_experts(_expert_inputs, expert_index, compute_n_expert, training=self.training)
                _expert_outputs = self.MoeFeedForward(_expert_inputs, expert_index, compute_n_expert, training=self.training)
                _combined_outputs = torch.einsum('gecm,gsec->gsm', _expert_outputs, _combine_array)
                combined_outputs = _combined_outputs if combined_outputs is None else combined_outputs + _combined_outputs
            combined_outputs = combined_outputs.reshape(*inputs.shape)
        return combined_outputs


class config:
    dim = 128
    base_emb_dim = 128
    num_experts_per_tok = 2
    expert_capacity_factor = 1.5
    min_group_size = 1
    router_z_loss_coef = 0.01
    aux_loss_coef = 0.01
    expert_chunk_size = 1
    mlp_activations = ['silu', 'linear']
    mgate = True
    mgate_dim = 44
    sfm_after_topn = True
    base_mlp_dim = 1408
    gate_noise_coef = 0.0
    init_weights_seed = 9876
    record_internal_nn_metrics = 0
    intermediate_size = 1408
    int8 = False
import pickle

inputs = torch.from_numpy(pickle.load(open('inputs.pkl', 'rb'))).to(torch.float16)
model = MoeBlock(config=config)
router_gate = torch.from_numpy(pickle.load(open('router_gate.pkl', 'rb')))
model.router_gate.data = router_gate
model.to(torch.float16)

MoeBlock(
  (MoeFeedForward): MoeFeedForward()
)

In [3]:
for k, v in model.named_parameters():
    print(k, v.shape, v.sum().item(), v.mean().item())

router_gate torch.Size([128, 8]) -3.546875 -0.0034637451171875
MoeFeedForward.wi_gate_0 torch.Size([8, 128, 1408]) 1441792.0 1.0
MoeFeedForward.wi_0 torch.Size([8, 128, 1408]) 1441792.0 1.0
MoeFeedForward.wo_0 torch.Size([8, 1408, 128]) 1441792.0 1.0
MoeFeedForward.mg torch.Size([8, 128, 44]) 45056.0 1.0


In [25]:
with torch.no_grad():
    outputs2 = model(inputs)

In [26]:
outputs2

tensor([[[4.9103e-02, 4.9103e-02, 4.9103e-02,  ..., 4.9103e-02,
          4.9103e-02, 4.9103e-02],
         [3.4820e+03, 3.4820e+03, 3.4820e+03,  ..., 3.4820e+03,
          3.4820e+03, 3.4820e+03],
         [1.9740e+03, 1.9740e+03, 1.9740e+03,  ..., 1.9740e+03,
          1.9740e+03, 1.9740e+03],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [9.0188e+01, 9.0188e+01, 9.0188e+01,  ..., 9.0188e+01,
          9.0188e+01, 9.0188e+01],
         [2.6140e+03, 2.6140e+03, 2.6140e+03,  ..., 2.6140e+03,
          2.6140e+03, 2.6140e+03]]], dtype=torch.float16)

In [19]:
jax_outputs = pickle.load(open('outputs.pkl', 'rb'))
jax_outputs = torch.from_numpy(jax_outputs.astype(np.float32))
torch_outputs = outputs.to(torch.float32)


In [21]:
torch.allclose(jax_outputs, torch_outputs)

True